In [0]:
import os
import json
JobEnv = False

dbutils.widgets.text("status", "")
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("data_ingestion_volume", "")
dbutils.widgets.text("databricks_host", "")
dbutils.widgets.text("job_orchestration_token", "")

is_job = False
# debugging for notbook parameters not being able to recieve from job-1 notebook
try:
    kvs = {k: dbutils.widgets.get(k) for k in ["status","catalog","schema","data_ingestion_volume","databricks_host","job_orchestration_token"]}
    print(f"Notebook running in job mode the configurations will be dynamically loaded from json file")
    print("Widget values:", kvs)
    status = dbutils.widgets.get("status")
    catalog = dbutils.widgets.get("catalog")
    schema = dbutils.widgets.get("schema")
    data_ingestion_volume = dbutils.widgets.get("data_ingestion_volume")
    data_ingestion_volume = dbutils.widgets.get("databricks_host")
    data_ingestion_volume = dbutils.widgets.get("job_orchestration_token")
    table_name = dbutils.widgets.get("table_name")
    is_job = True
except:
    is_job = False

In [0]:
if is_job:
  spark.sql(f"""
  CREATE OR REPLACE TABLE {catalog}.{schema}.{table_name} AS
              SELECT ct.customer_id, ct.first_name, ct.last_name, ct.phone, st.sale_id, st.sale_date, st.sales_amount, st.payment_method, st.state FROM {catalog}.{schema}.customers_table AS ct INNER JOIN {catalog}.{schema}.sales_table AS st ON ct.customer_id = st.customer_id ORDER BY customer_id DESC
            """)
else:
  print(f"Notebook is running on interactive mode skip joining customers_table with sales_table!")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
if is_job:
  df = spark.sql(f"""
              SELECT * FROM {catalog}.{schema}.{table_name}
            """)


In [0]:
if is_job:
    total_count = df.count()
    distinct_count = df.dropDuplicates().count()

    has_duplicates = total_count > distinct_count
    print(f"has_duplicates ---> {has_duplicates}")

    import json

    output = {
        "catalog": catalog,
        "schema": schema,
        "customers_sales_table_has_duplicates": has_duplicates,
        "table_name":table_name,
        "gold_table_name":dbutils.widgets.get("customers_sales_table_gold")
    }

    dbutils.jobs.taskValues.set(key="metadata", value=json.dumps(output))
    dbutils.jobs.taskValues.set(key="has_duplicates", value=has_duplicates)
    print(f"Notebook running on Job mode sending metadata to another task : {output}")
else:
    print(f"Notebook running on Interactive mode failed to send metadata to another task : {output}")